In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.datos import cargar_obt, agregar_features

df = cargar_obt()
df = agregar_features(df)

df[["sales_mean_lag42_28d", "sales_mean_lag42_dow"]].describe().round(1)

,sales_mean_lag42_28d,sales_mean_lag42_dow
count,960875.0,827323.0
mean,6944.5,6963.4
std,2508.7,2751.0
min,520.0,286.0
25%,5259.1,5157.0
50%,6542.4,6528.2
75%,8057.0,8211.4
max,30795.5,38037.0


In [2]:
df_mod = df[df["is_model_eligible"]]

cols = ["sales", "sales_mean_lag42_28d", "sales_mean_lag42_dow", "is_promo", "day_of_week_iso"]
print(df_mod[cols].astype(float).corr(method="spearman")["sales"].round(3))

print("\nFaltantes en días elegibles:")
print(df_mod[["sales_mean_lag42_28d", "sales_mean_lag42_dow"]].isna().sum())

sales                   1.000
sales_mean_lag42_28d    0.721
sales_mean_lag42_dow    0.793
is_promo                0.420
day_of_week_iso        -0.179
Name: sales, dtype: float64

Faltantes en días elegibles:
sales_mean_lag42_28d    46489
sales_mean_lag42_dow    47038
dtype: int64


In [3]:
import numpy as np

corte = pd.Timestamp("2015-03-01")
features = ["sales_mean_lag42_28d", "sales_mean_lag42_dow"]

df_oculto = df.drop(columns=features).copy()
df_oculto.loc[df_oculto["sales_date"] > corte, "sales"] = np.nan
df_oculto = agregar_features(df_oculto)

def iguales(desde, hasta):
    m = (df["sales_date"] > desde) & (df["sales_date"] <= hasta)
    a = df.loc[m, features].to_numpy(dtype=float)
    b = df_oculto.loc[m, features].to_numpy(dtype=float)
    return np.allclose(a, b, equal_nan=True)

print("Días 1 a 42 después del corte, iguales:", iguales(corte, corte + pd.Timedelta(days=42)))
print("Días 43 a 84 después del corte, iguales:", iguales(corte + pd.Timedelta(days=42), corte + pd.Timedelta(days=84)))

Días 1 a 42 después del corte, iguales: True
Días 43 a 84 después del corte, iguales: False


## Features derivadas

Ambas usan solo ventas de días elegibles con al menos 42 días de rezago (el horizonte máximo del pronóstico). Se calculan en `src/datos.py` (`agregar_features`).

| Feature | Definición | Origen | Evidencia (Spearman con ventas) | ¿Existe al predecir? | Codificación |
|---|---|---|---|---|---|
| `sales_mean_lag42_28d` | Media de ventas de la tienda en los días -69 a -42 | Hallazgo 3 del EDA | 0,721 | Sí: usa ventas de hace 42 días o más | Numérica, no aplica |
| `sales_mean_lag42_dow` | Media de ventas de la tienda en el mismo día de la semana, días -42, -49, -56 y -63 | Patrón semanal de la estadística descriptiva | 0,793 (mayor que la media de la OBT con fuga, 0,747) | Sí: usa ventas de hace 42 días o más | Numérica, no aplica |

**Prueba de ausencia de fuga:** se borraron las ventas posteriores al 2015-03-01 y se recalcularon las features. En los 42 días siguientes al corte son idénticas a las calculadas con todos los datos (no usan información futura); a partir del día 43 cambian (control: la prueba detecta dependencia del futuro cuando existe).

**Faltantes:** cerca del 5,5% de los días elegibles no tiene 42 días de historia (inicio de 2013 y regreso del hueco de 2014). Se excluyen del entrenamiento: en operación, una tienda existente siempre tiene esa historia, y rellenarlos inventaría datos.

**Columnas de la OBT que no se usan por fuga:** `customers`, `sales_per_customer`, `is_open_zero_sales`, `is_sales_without_customers` (calculadas con las ventas del mismo día) y `sales_mean_prev_7d`, `sales_mean_prev_28d`, `customers_mean_prev_28d` (usan ventas de días que no existen a 42 días). Las columnas del día anterior y siguiente (`was_open_prev_day`, `was_promo_prev_day`, `is_open_next_day`, feriados) sí se usan: provienen del calendario de aperturas, promociones y feriados, que se conoce con anticipación.

In [5]:
import numpy as np
from src.pipeline import preparar_datos, crear_preprocesador, pliegues_temporales, FEATURES

entrenamiento, prueba = preparar_datos(df)
print("Entrenamiento:", entrenamiento.shape,
      entrenamiento["sales_date"].min().date(), "a", entrenamiento["sales_date"].max().date())
print("Prueba:", prueba.shape,
      prueba["sales_date"].min().date(), "a", prueba["sales_date"].max().date())

for i, (ent, val) in enumerate(pliegues_temporales(entrenamiento), 1):
    print(f"Pliegue {i}: entrena {len(ent):,} filas, valida {len(val):,} filas")

prep = crear_preprocesador()
X = prep.fit_transform(entrenamiento[FEATURES])
print("Matriz preprocesada:", X.shape, "| vacíos:", np.isnan(X).sum())

Entrenamiento: (757018, 59) 2013-02-12 a 2015-06-19
Prueba: (40282, 59) 2015-06-20 a 2015-07-31
Pliegue 1: entrena 642,608 filas, valida 39,969 filas
Pliegue 2: entrena 682,577 filas, valida 36,970 filas
Pliegue 3: entrena 719,547 filas, valida 37,471 filas
Matriz preprocesada: (757018, 53) | vacíos: 0
